# Build a Debugging Tutor with Gradio

In this notebook you build a small debugging tutor and study how a chatbot is assembled from prompts, examples, memory, and a user interface.

By the end, you should be able to explain:
- how the system prompt changes the tutor's behavior
- how a few-shot example nudges the reply style
- how older turns can be compressed into a short history
- how model choice and reasoning settings affect the final answer

As you work through the notebook, try changing **one setting at a time** so you can clearly see what changed and why.


In [3]:
# %pip -q install gradio llama-cpp-python

import gradio as gr
from llama_cpp import Llama
import time
import os
import warnings

warnings.filterwarnings("ignore")

## 1. Load Local Language Model
First, we load the quantized language model into local memory. The `context_window` parameter defines the maximum number of tokens the model can process simultaneously.


In [ ]:
# Change this to match where your local GGUF model is stored.
# model_file_path = "/home/jovyan/Small_Models_SP26/Seoha/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf"
# model_file_path = "/home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf"
model_file_path = "/home/jovyan/shared/DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf"

# Change this to increase or decrease the context window.
context_window = 1024

local_model = None
local_model_name = model_file_path.split("/")[-1].replace(".gguf", "")

try:
    local_model = Llama(
        model_path=model_file_path,
        n_ctx=context_window,
        n_threads=4,
        verbose=False,
    )
    print("Local model loaded:", local_model_name)
except Exception as error:
    print("Local model failed to load.")
    print(type(error).__name__ + ":", error)


## 1.1 Connect to Cloud API (Optional)
We can also connect to a Cloud API. This allows us to compare our small local model against massive cloud models. If you do not have an API key, the chatbot will simply use the local model.

In [1]:
# Keep the notebook working even if no API is enabled.
api_client = None
api_models = {}

### 1.1.A Option: Groq
1. Get a free [keys](https://console.groq.com/keys)
2. Check models at [free plan limits](https://console.groq.com/docs/rate-limits)

In [4]:
# ── UNCOMMENT THIS CELL TO ENABLE GROQ ──
# Uncomment next line if ModuleNotFoundError: No module named 'groq', or 'dotenv'
# %pip -q install groq python-dotenv 
from groq import Groq
from dotenv import load_dotenv

load_dotenv(".env")  # Change path if your .env is elsewhere
api_key = os.getenv("GROQ_API_KEY") # create file with .env -> paste the key GROQ_API_KEY=gsk_...
print("Groq API Key:", "loaded" if api_key else "NOT FOUND")

if api_key:
    api_client = Groq(api_key=api_key, base_url="https://api.groq.com/openai/v1")

    # Add or remove models here.
    api_models = {
        "GPT-OSS 20B": "openai/gpt-oss-20b",
        "GPT-OSS 120B": "openai/gpt-oss-120b",
    }
    print("Groq models loaded:", list(api_models.keys()))

Groq API Key: loaded
Groq models loaded: ['GPT-OSS 20B', 'GPT-OSS 120B']


### 1.1.B Option: OpenAI

In [23]:
# ── UNCOMMENT THIS CELL TO ENABLE OPENAI ──
# # %pip -q install openai python-dotenv # Uncomment this line if ModuleNotFoundError: No module named 'openai', or 'dotenv'
# from openai import OpenAI
# from dotenv import load_dotenv

# load_dotenv(".env")  # Change path if your .env is elsewhere
# api_key = os.getenv("OPENAI_API_KEY") # .env -> OPENAI_API_KEY=sk-...
# print("OpenAI API Key:", "loaded" if api_key else "NOT FOUND")

# if api_key:
#     api_client = OpenAI(api_key=api_key)

#     # Add or remove models here.
#     api_models = {
#         "gpt-4o-mini": "gpt-4o-mini",
#     }
#     print("OpenAI models loaded:", list(api_models.keys()))

## 2. Blueprint

The blueprint is the easiest part of the notebook to modify.

This is where you can:
- change the AI personality.
- add or remove a short reasoning instruction
- add a few-shot example
- design short test cases for classroom experiments

If you want to reuse this notebook for another chatbot, this cell is usually the first place to edit.


In [5]:
# Edit this cell to customize your own AI assistant.
app_title = "AI Debugging Tutor"
app_desc = "Experiment with prompting, memory, compression, and reasoning."

system_prompt = """
You are a Socratic debugging tutor.

Rules:
- Do not give the final answer or full corrected code.
- Ask one small guiding question at a time.
- Give only the next hint or check.
- Keep answers short and clear.
- If the student asks for the answer, do not give it.
- End with one short question.
"""

reasoning_prompt = """
Keep reasoning brief.
Always provide a short final response.
Do not spend more than 450 tokens on internal reasoning.
"""

few_shot_example = """
Example 1:
Student:
age = 25
print("I am " + age)

Assistant:
What is the type of `age` right now? Can `+` join a string and an integer directly?

Example 2:
Student:
Please just give me the answer.

Assistant:
Before we jump to the answer, what part feels most confusing right now: the error message, the variable type, or the loop logic?
"""

test_cases = {
    "1. CSV Loading [Concept]": {
        "input": """How do I read a CSV file with the datascience package?
Please guide me step by step."""
    },
    "2. TypeError [Easy]": {
        "input": """age = 25
print("I am " + age)

TypeError: can only concatenate str to str

Please ask me one small question at a time."""
    },
    "3. Average Score [Logic]": {
        "input": """def average_score(scores):
    total = 0
    for score in scores:
        total = score
    return total / len(scores)

print(average_score([80, 90, 70, 100]))

Please help me find the logic bug step by step."""
    },
    "4. Prefix Sum [Reasoning]": {
        "input": """prefix = [3, 4, 8, 9, 14]

def range_sum(prefix, left, right):
    if left == 0:
        return prefix[right]
    return prefix[right] - prefix[left]

print(range_sum(prefix, 1, 3))  # expected 6

Help me reason about the indexing."""
    },
    "5. Table Filter [Reasoning]": {
        "input": """scores.where("Score" > 80)

This does not work the way I expected.
Help me figure out what kind of input `where` wants."""
    }
}

print("Blueprint loaded.")


Blueprint loaded.


## 3. Memory and Compression

A chatbot does not magically remember the past. It only sees the text we send in the current prompt.

This notebook uses two simple memory ideas:
1. keep a small number of recent turns in detail
2. compress older turns into a short history summary

This pattern is easy to understand, easy to modify, and reusable in many chatbot projects.


In [6]:
run_log = []
conversation_summary = ""

## 4. Chatbot Engine

The engine is split into small steps so each cell has one clear job.

This makes the notebook easier to read, modify, and reuse for another chatbot.


### 4.1 Clean messages for reuse

When we reuse past messages in a prompt, we want plain text.

This helper removes UI-only extras and keeps only the role and content.


In [7]:
def clean_message(message):
    """Return a simple {role, content} message dictionary."""
    role = message.get("role", "user")
    content = message.get("content", "")

    if isinstance(content, str):
        text = content
    elif isinstance(content, dict):
        text = content.get("text", str(content))
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        text = "".join(parts)
    else:
        text = str(content)

    if role == "assistant":
        text = text.split("\n\n`Time: ")[0]

    return {"role": role, "content": text}


def build_summary_source(old_history):
    """Turn older messages into a single text block for summarization."""
    lines = []

    for message in old_history:
        clean_item = clean_message(message)
        role = clean_item["role"].upper()
        content = clean_item["content"]
        lines.append("[" + role + "] " + content)

    return "\n".join(lines)


### 4.2 Build the prompt and choose the current input

This step decides:
- which instructions go into the system prompt
- whether reasoning instructions are added
- whether few-shot examples are added
- which text becomes the current user query


In [8]:
def build_prompt(system_prompt, reasoning_prompt, few_shot_example, use_reasoning, use_few_shot):
    """Build the active system prompt for the current run."""
    active_prompt = system_prompt.strip()

    if use_reasoning:
        active_prompt += "\n\n" + reasoning_prompt.strip()

    if use_few_shot:
        active_prompt += "\n\n" + few_shot_example.strip()

    return active_prompt


def build_query(user_input, test_case, chat_history):
    """Use the textbox text, or load the selected test case on the first turn."""
    if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
        if user_input.strip() != "":
            return user_input.strip()
        return test_cases[test_case]["input"]

    return user_input.strip()


def split_history(chat_history, memory_turns):
    """Keep recent turns in detail and move older turns into the summary bucket."""
    keep_messages = int(memory_turns) * 2  # one turn = one user + one assistant message

    if memory_turns > 0 and len(chat_history) > keep_messages + 4:
        old_history = chat_history[:-keep_messages]
        recent_history = chat_history[-keep_messages:]
    else:
        old_history = []
        recent_history = chat_history

    return old_history, recent_history


def count_text_tokens(text):
    """Estimate token count. Use the local tokenizer when available."""
    if local_model is not None:
        try:
            return len(local_model.tokenize(text.encode("utf-8")))
        except Exception:
            pass

    return max(1, len(text.split()))


### 4.3 Compress old history and assemble the context

Older turns are summarized so the prompt stays short enough for the model.

This cell is useful for experiments:
- try different memory lengths
- inspect the summary in the **History** box
- compare the **Context** box before and after compression

Notice the main idea: recent turns stay detailed, while older turns become a shorter summary.


In [9]:
def summarize_history(old_history, model_name):
    """Summarize older turns into a short history string."""
    global conversation_summary

    if not old_history:
        return

    summary_source = build_summary_source(old_history)
    summary_messages = [
        {
            "role": "system",
            "content": (
                "Summarize this older conversation in 4 short bullet points. "
                "Keep the student's goal, important mistakes, and useful facts. "
                "Do not answer the student."
            ),
        },
        {"role": "user", "content": summary_source},
    ]

    try:
        if model_name in api_models and api_client is not None:
            model_id = api_models[model_name]
            response = api_client.chat.completions.create(
                model=model_id,
                messages=summary_messages,
                temperature=0.2,
                max_completion_tokens=160,
            )
            summary_text = response.choices[0].message.content

        elif local_model is not None:
            response = local_model.create_chat_completion(
                messages=summary_messages,
                temperature=0.2,
                max_tokens=160,
                top_p=0.95,
            )
            summary_text = response["choices"][0]["message"]["content"]

        else:
            return

        if summary_text is not None:
            conversation_summary = summary_text.strip()

    except Exception:
        # If summary creation fails, keep the previous summary and continue.
        return


def build_messages(active_prompt, chat_history, memory_turns, model_name):
    """Build the final message list that will be sent to the model."""
    messages = [{"role": "system", "content": active_prompt}]
    old_history, recent_history = split_history(chat_history, memory_turns)

    if old_history:
        summarize_history(old_history, model_name)

    if conversation_summary != "":
        messages.append({
            "role": "system",
            "content": "Conversation summary from older turns:\n" + conversation_summary,
        })

    if memory_turns > 0:
        for message in recent_history:
            messages.append(clean_message(message))

    return messages


def build_context_text(messages):
    """Create a readable text view of the exact prompt context."""
    parts = []

    for message in messages:
        parts.append("[" + message["role"].upper() + "]")
        parts.append(str(message["content"]))
        parts.append("----------------------------------------")

    return "\n".join(parts)


def build_history_text():
    """Show the current compressed history summary."""
    if conversation_summary == "":
        return "No compressed history yet."
    return conversation_summary


### 4.4 Generate a response

For simplicity and stability, this notebook uses a normal completion call instead of token-by-token streaming.

When reasoning is enabled, the notebook:
- adds a short reasoning instruction to the prompt
- applies a reasoning level only for supported GPT-OSS models
- hides raw reasoning details from the student-facing interface

This keeps the UI simple while still letting you experiment with reasoning settings.


In [10]:
def supports_reasoning(model_name):
    """Return True for models that support Groq reasoning settings."""
    return model_name in ["GPT-OSS 20B", "GPT-OSS 120B"]


def get_reasoning_args(model_name, use_reasoning, reasoning_level):
    """Return the correct API parameters for supported reasoning models."""
    if not supports_reasoning(model_name):
        return {}

    if not use_reasoning:
        return {"include_reasoning": False}

    return {
        "reasoning_effort": reasoning_level.lower(),
        "include_reasoning": False,
    }


def generate_response(messages, model_name, temperature, max_tokens, use_reasoning, reasoning_level):
    """Generate a reply and return text plus token usage when available."""
    if model_name in api_models and api_client is not None:
        model_id = api_models[model_name]
        request_args = {
            "model": model_id,
            "messages": messages,
            "temperature": float(temperature),
            "max_completion_tokens": int(max_tokens),
            "top_p": 0.95,
        }
        request_args.update(get_reasoning_args(model_name, use_reasoning, reasoning_level))

        response = api_client.chat.completions.create(**request_args)
        final_text = response.choices[0].message.content or ""

        prompt_tokens = None
        output_tokens = None
        usage = getattr(response, "usage", None)

        if usage is not None:
            prompt_tokens = getattr(usage, "prompt_tokens", None)
            output_tokens = getattr(usage, "completion_tokens", None)

        if output_tokens is None:
            output_tokens = count_text_tokens(final_text)

        return final_text, prompt_tokens, output_tokens

    if local_model is not None:
        response = local_model.create_chat_completion(
            messages=messages,
            temperature=float(temperature),
            max_tokens=int(max_tokens),
            top_p=0.95,
        )
        final_text = response["choices"][0]["message"]["content"] or ""

        usage = response.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens")
        output_tokens = usage.get("completion_tokens")

        if output_tokens is None:
            output_tokens = count_text_tokens(final_text)

        return final_text, prompt_tokens, output_tokens

    final_text = "No model is available. Load the local model or enable a Groq API model."
    return final_text, None, count_text_tokens(final_text)


### 4.5 Run the chatbot and keep a log

This final step ties everything together:
- build the prompt
- build the context
- call the model
- update the chat
- update the compressed history
- store a short experiment log

This is the main function to study if you want to adapt the notebook into another chatbot.


In [11]:
def build_log_rows(run_log):
    """Return the last 10 runs for the experiment table."""
    log_rows = []

    for record in run_log[-10:]:
        log_rows.append([
            record["run"],
            record["scenario"],
            record["model"],
            record["reasoning"],
            record["input_tokens"],
            record["output_tokens"],
            record["latency"],
        ])

    return log_rows


def run_chatbot(
    user_input,
    system_prompt,
    reasoning_prompt,
    few_shot_example,
    use_reasoning,
    reasoning_level,
    use_few_shot,
    memory_turns,
    temperature,
    max_tokens,
    test_case,
    model_name,
    chat_history,
):
    """Run one full chatbot turn and return all UI outputs."""
    global run_log

    if chat_history is None:
        chat_history = []

    active_prompt = build_prompt(
        system_prompt,
        reasoning_prompt,
        few_shot_example,
        use_reasoning,
        use_few_shot,
    )
    messages = build_messages(active_prompt, chat_history, memory_turns, model_name)
    user_query = build_query(user_input, test_case, chat_history)
    history_text = build_history_text()

    if user_query == "":
        return chat_history, "", history_text, [], "", "**Status:** Waiting for input"

    messages.append({"role": "user", "content": user_query})
    context_text = build_context_text(messages)

    display_user = user_input.strip()
    if display_user == "":
        if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
            display_user = test_cases[test_case]["input"]
        else:
            display_user = "[" + test_case + "]"

    # Copy the list so Gradio state updates do not mutate the previous object in place.
    chat_history = list(chat_history)
    chat_history.append({"role": "user", "content": display_user})
    chat_history.append({"role": "assistant", "content": ""})

    start_time = time.perf_counter()
    final_text, api_prompt_tokens, output_tokens = generate_response(
        messages,
        model_name,
        temperature,
        max_tokens,
        use_reasoning,
        reasoning_level,
    )
    latency = round(time.perf_counter() - start_time, 2)

    # Prefer API usage when available. Otherwise estimate from the text we built locally.
    input_tokens = api_prompt_tokens if api_prompt_tokens is not None else count_text_tokens(context_text)

    badge = (
        "\n\n`Time: " + str(latency)
        + "s | Context: " + str(round((input_tokens / max(1, context_window)) * 100, 1))
        + "% | Input tokens: " + str(input_tokens)
        + " | Output tokens: " + str(output_tokens) + "`"
    )

    chat_history[-1]["content"] = final_text + badge

    run_record = {
        "run": len(run_log) + 1,
        "scenario": test_case,
        "model": model_name,
        "reasoning": "On" if use_reasoning else "Off",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
    }
    run_log.append(run_record)

    log_rows = build_log_rows(run_log)
    history_text = build_history_text()

    return chat_history, context_text, history_text, log_rows, "", "**Status:** Done"


def clear_chat():
    """Reset chat history, compressed history, and the experiment log."""
    global conversation_summary
    global run_log

    conversation_summary = ""
    run_log = []

    return [], "", "No compressed history yet.", [], "", "**Status:** Ready"


def load_test_case(test_case):
    """Fill the input box with the selected test case."""
    if test_case == "Free Typing":
        return "", "**Status:** Ready"

    return test_cases[test_case]["input"], "**Status:** Loaded " + test_case


def toggle_reasoning_level(use_reasoning):
    """Show the reasoning-level control only when reasoning is enabled."""
    return gr.update(visible=use_reasoning)


def run_chatbot_ui(
    user_input,
    system_prompt,
    reasoning_prompt,
    few_shot_example,
    use_reasoning,
    reasoning_level,
    use_few_shot,
    memory_turns,
    temperature,
    max_tokens,
    test_case,
    model_name,
    chat_state,
):
    """Small wrapper that keeps chat state separate from the Chatbot component."""
    chat_history, context_text, history_text, log_rows, next_input, status_message = run_chatbot(
        user_input,
        system_prompt,
        reasoning_prompt,
        few_shot_example,
        use_reasoning,
        reasoning_level,
        use_few_shot,
        memory_turns,
        temperature,
        max_tokens,
        test_case,
        model_name,
        chat_state,
    )

    return (
        chat_history,
        context_text,
        history_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )


def clear_chat_ui():
    """Reset both the visible chat and the hidden chat state."""
    chat_history, context_text, history_text, log_rows, next_input, status_message = clear_chat()

    return (
        chat_history,
        context_text,
        history_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )


## 5. Build the Interface

The final layout keeps the main interaction simple:
- chat on the left
- controls on the right
- **Context** and **History** underneath
- experiment log at the bottom

This layout is practical for learning because students can chat normally, then look below to inspect what the model received and how old turns were compressed.


In [12]:
ui_scenarios = ["Free Typing"] + list(test_cases.keys())

ui_models = []
if local_model_name:
    ui_models.append(local_model_name)
ui_models.extend(list(api_models.keys()))

with gr.Blocks(title=app_title) as web_app:
    gr.Markdown("# " + app_title)
    gr.Markdown("*" + app_desc + "*")

    chat_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=7):
            chat_window = gr.Chatbot(
                label="Chat",
                height=535,
            )

            with gr.Row():
                user_input_box = gr.Textbox(
                    show_label=False,
                    placeholder="Type your question here...",
                    lines=4,
                    scale=8,
                )
                with gr.Column(scale=1, min_width=60):
                    ask_button = gr.Button("Ask", variant="primary")
                    clear_button = gr.Button("New Chat")

        with gr.Column(scale=4):
            test_case_dropdown = gr.Dropdown(
                choices=ui_scenarios,
                value="Free Typing",
                label="Test Case",
            )

            model_dropdown = gr.Dropdown(
                choices=ui_models,
                value=ui_models[0] if ui_models else None,
                label="Model",
            )

            use_reasoning = gr.Checkbox(
                label="Use Reasoning (GPT OSS)",
                value=False,
            )

            reasoning_level = gr.Dropdown(
                choices=["Low", "Medium", "High"],
                value="Low",
                label="Reasoning Level",
                visible=False,
            )

            use_few_shot = gr.Checkbox(
                label="Use Few-Shot",
                value=False,
            )

            with gr.Accordion("Model Settings", open=True):
                memory_turns_slider = gr.Slider(
                    minimum=0,
                    maximum=8,
                    step=1,
                    value=1,
                    label="Memory Turns",
                )

                temperature_slider = gr.Slider(
                    minimum=0.0,
                    maximum=1.0,
                    step=0.1,
                    value=0.2,
                    label="Temperature",
                )

                max_tokens_slider = gr.Slider(
                    minimum=64,
                    maximum=1024,
                    step=64,
                    value=512,
                    label="Max Tokens",
                )

            with gr.Accordion("Edit Prompts", open=False):
                system_prompt_box = gr.Textbox(
                    label="System Prompt",
                    value=system_prompt,
                    lines=8,
                )

                reasoning_prompt_box = gr.Textbox(
                    label="Reasoning Prompt",
                    value=reasoning_prompt,
                    lines=3,
                )

                few_shot_box = gr.Textbox(
                    label="Few-Shot Example",
                    value=few_shot_example,
                    lines=8,
                )

    with gr.Row():
        context_box = gr.Textbox(
            label="Context",
            value="",
            lines=14,
            interactive=False,
        )

        history_box = gr.Textbox(
            label="History",
            value="No compressed history yet.",
            lines=14,
            interactive=False,
        )

    log_table = gr.Dataframe(
        headers=["Run", "Scenario", "Model", "Reasoning", "Input", "Output", "Latency"],
        datatype=["number", "str", "str", "str", "number", "number", "number"],
        row_count=10,
        col_count=(7, "fixed"),
        interactive=False,
        label="Experiment Log",
    )

    status_box = gr.Markdown("**Status:** Ready")

    input_list = [
        user_input_box,
        system_prompt_box,
        reasoning_prompt_box,
        few_shot_box,
        use_reasoning,
        reasoning_level,
        use_few_shot,
        memory_turns_slider,
        temperature_slider,
        max_tokens_slider,
        test_case_dropdown,
        model_dropdown,
        chat_state,
    ]

    output_list = [
        chat_window,
        context_box,
        history_box,
        log_table,
        user_input_box,
        status_box,
        chat_state,
    ]

    use_reasoning.change(
        fn=toggle_reasoning_level,
        inputs=use_reasoning,
        outputs=reasoning_level,
    )

    test_case_dropdown.change(
        fn=load_test_case,
        inputs=test_case_dropdown,
        outputs=[user_input_box, status_box],
    )

    ask_button.click(
        fn=run_chatbot_ui,
        inputs=input_list,
        outputs=output_list,
    )

    user_input_box.submit(
        fn=run_chatbot_ui,
        inputs=input_list,
        outputs=output_list,
    )

    clear_button.click(
        fn=clear_chat_ui,
        inputs=None,
        outputs=output_list,
    )

# Launch the website
web_app.launch(share=True, inline=True)


NameError: name 'local_model_name' is not defined